# CDR Data Preprocessing
Reads Call Detail Record files dropped into `data/cdr_incoming/` (the SFTP landing folder) and runs them through cleaning, encoding, and scaling before they reach the anomaly detection / forecasting / clustering models. Re-run this notebook top to bottom whenever new CDR files have landed.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path.cwd().parent / "data"
INCOMING_DIR = DATA_DIR / "cdr_incoming"
PROCESSED_DIR = DATA_DIR / "processed"
ARCHIVE_DIR = DATA_DIR / "archive"

OUTLIER_IQR_MULTIPLIER = 1.5
REDUNDANCY_CORR_THRESHOLD = 0.9

RAW_COLUMNS = {
    "date": "date",
    "orig. time": "orig_time",
    "orig time": "orig_time",
    "orig.": "orig",
    "orig": "orig",
    "dest.": "dest",
    "dest": "dest",
    "call classification": "call_classification",
    "duration (sec)": "duration_sec",
    "duration": "duration_sec",
    "charge": "charge",
}
REQUIRED_COLUMNS = set(RAW_COLUMNS.values())
SUPPORTED_EXTENSIONS = {".csv", ".txt", ".tsv"}


## Load CDR files
Every supported file currently sitting in `cdr_incoming/` gets read and concatenated. Files that don't match the expected CDR schema (e.g. a scanned PDF with no machine-readable table) are skipped with a warning rather than failing the whole run, since the folder accumulates files of mixed provenance over time.

In [3]:
def derive_cucm_fields(raw: pd.DataFrame) -> pd.DataFrame:
    """Map a real Cisco CallManager CDR export onto the columns normalize_columns expects.

    This export's own column names (dateTimeOrigination, callingPartyNumber,
    finalCalledPartyNumber, duration, ...) don't match RAW_COLUMNS, and it has
    no call-classification/charge columns at all since it's an internal
    cluster's own CDRs, not a billed telco feed.
    """
    cols = {c.strip().lower(): c for c in raw.columns}
    if "datetimeorigination" not in cols or "callingpartynumber" not in cols:
        return raw

    raw = raw.copy()
    origination = pd.to_datetime(pd.to_numeric(raw[cols["datetimeorigination"]], errors="coerce"), unit="s")
    raw["date"] = origination.dt.strftime("%Y-%m-%d")
    raw["orig_time"] = origination.dt.strftime("%H:%M:%S")
    raw["orig"] = raw[cols["callingpartynumber"]]
    dest_col = cols.get("finalcalledpartynumber") or cols.get("originalcalledpartynumber")
    raw["dest"] = raw[dest_col]
    if "duration" in cols:
        raw["duration_sec"] = raw[cols["duration"]]
        raw = raw.drop(columns=[cols["duration"]])
    raw["charge"] = 0  # CUCM CDRs carry no billing charge; internal calls bill at 0
    raw["call_classification"] = "Internal"  # this export is an internal cluster's own CDRs
    return raw


def normalize_columns(raw: pd.DataFrame) -> pd.DataFrame:
    raw = derive_cucm_fields(raw)
    raw = raw.rename(columns={c: RAW_COLUMNS.get(c.strip().lower(), c.strip().lower()) for c in raw.columns})
    missing = REQUIRED_COLUMNS - set(raw.columns)
    if missing:
        raise ValueError(f"missing expected columns: {sorted(missing)}")
    return raw[list(REQUIRED_COLUMNS)]


def read_one(path: Path) -> pd.DataFrame:
    sep = "\t" if path.suffix.lower() == ".tsv" else ","
    file_df = pd.read_csv(path, sep=sep, dtype=str)
    file_df = normalize_columns(file_df)
    file_df["source_file"] = path.name
    return file_df


files = sorted(p for p in INCOMING_DIR.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS)
frames = []
for path in files:
    try:
        frames.append(read_one(path))
    except (ValueError, pd.errors.ParserError) as exc:
        print(f"skipping {path.name}: {exc}")

df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=list(REQUIRED_COLUMNS) + ["source_file"])
print(f"loaded {len(df)} records from {len(frames)} file(s)")
print(df.head(10))


loaded 0 records from 0 file(s)
Empty DataFrame
Columns: [dest, duration_sec, orig, date, orig_time, call_classification, charge, source_file]
Index: []


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   dest                 0 non-null      object
 1   duration_sec         0 non-null      object
 2   orig                 0 non-null      object
 3   date                 0 non-null      object
 4   orig_time            0 non-null      object
 5   call_classification  0 non-null      object
 6   charge               0 non-null      object
 7   source_file          0 non-null      object
dtypes: object(8)
memory usage: 124.0+ bytes


## Missing values
QoS is never populated by the telco (always `NA`) — drop it rather than impute.

In [5]:
df = df.drop(columns=["qos"], errors="ignore")
df.columns.tolist()
df.head()

,dest,duration_sec,orig,date,orig_time,call_classification,charge,source_file


## Noise
Parse Date + Orig. Time into a single timestamp, coerce Duration/Charge to numeric, and drop records that are physically invalid: negative durations or charges, or timestamps that couldn't be parsed at all.

In [4]:
df["call_timestamp"] = pd.to_datetime(df["date"].astype(str) + " " + df["orig_time"].astype(str), errors="coerce")
df["duration_sec"] = pd.to_numeric(df["duration_sec"], errors="coerce")
df["charge"] = pd.to_numeric(df["charge"], errors="coerce")

before = len(df)
df = df.dropna(subset=["call_timestamp"])
df = df[(df["duration_sec"].fillna(0) >= 0) & (df["charge"].fillna(0) >= 0)]
print(f"dropped {before - len(df)} noisy record(s), {len(df)} remain")
df.head()

dropped 0 noisy record(s), 0 remain


,date,orig,dest,orig_time,duration_sec,call_classification,charge,source_file,call_timestamp


## Outliers
Flag statistical outliers in call duration via IQR fencing — **not removed**, since these flagged rows are exactly what the Isolation Forest anomaly detection model needs to see downstream.

In [5]:
q1, q3 = df["duration_sec"].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - OUTLIER_IQR_MULTIPLIER * iqr, q3 + OUTLIER_IQR_MULTIPLIER * iqr
df["is_outlier"] = ~df["duration_sec"].between(lower, upper)
print(f"{df['is_outlier'].sum()} of {len(df)} records flagged as duration outliers")
df[df["is_outlier"]].head()

0 of 0 records flagged as duration outliers


,date,orig,dest,orig_time,duration_sec,call_classification,charge,source_file,call_timestamp,is_outlier


## Redundant columns
Compute the Pearson correlation matrix across numeric columns and drop one column from any pair that's redundant (correlated above the threshold).

In [6]:
numeric = df.select_dtypes(include="number")
corr = numeric.corr(method="pearson").abs()
corr

,duration_sec,charge
duration_sec,NaN,NaN
charge,NaN,NaN


In [7]:
to_drop = set()
for i, col_a in enumerate(corr.columns):
    for col_b in corr.columns[i + 1:]:
        if col_a in to_drop or col_b in to_drop:
            continue
        if corr.loc[col_a, col_b] > REDUNDANCY_CORR_THRESHOLD:
            to_drop.add(col_b)

print(f"dropping redundant column(s): {sorted(to_drop) or 'none'}")
df = df.drop(columns=list(to_drop))

dropping redundant column(s): none


## Encoding
One-hot encode Call Classification (Internal / Others / ...).

In [8]:
df = pd.get_dummies(df, columns=["call_classification"], prefix="call_class")
df.head()

,date,orig,dest,orig_time,duration_sec,charge,source_file,call_timestamp,is_outlier


## Feature scaling
Duration has a long right tail (a handful of very long calls). Log-transform it toward a bell curve, then standardize to zero mean / unit variance.

In [9]:
log_duration = np.log1p(df["duration_sec"].fillna(0))
std = log_duration.std()
df["duration_sec_scaled"] = (log_duration - log_duration.mean()) / std if std else 0.0
df[["duration_sec", "duration_sec_scaled"]].describe()

,duration_sec,duration_sec_scaled
count,0.0,0.0
mean,NaN,NaN
std,NaN,NaN
min,NaN,NaN
25%,NaN,NaN
50%,NaN,NaN
75%,NaN,NaN
max,NaN,NaN


## Feature selection
Drop Charge — it isn't predictive for usage-pattern analysis (nearly all internal calls bill at 0) and was only kept around for the redundancy check above. Call records are **not** deduplicated: the same two extensions can legitimately call each other multiple times a day, so identical rows are real, distinct events, not duplicates to collapse.

In [10]:
df = df.drop(columns=["charge"], errors="ignore")
df.head()

,date,orig,dest,orig_time,duration_sec,source_file,call_timestamp,is_outlier,duration_sec_scaled


## Save
Append the cleaned batch to the processed dataset, then move the ingested files out of `cdr_incoming/` so the next run only picks up newly arrived files.

In [13]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
out_path = PROCESSED_DIR / "cdr_preprocessed.csv"
header = not out_path.exists()
df.to_csv(out_path, mode="a", header=header, index=False)

ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
for path in files:
    path.rename(ARCHIVE_DIR / path.name)

print(f"wrote {len(df)} records -> {out_path}")
print(df.head(10))

wrote 0 records -> /home/mugo/Projects/TeleSpy/ML/data/processed/cdr_preprocessed.csv
Empty DataFrame
Columns: [date, orig, dest, orig_time, duration_sec, source_file, call_timestamp, is_outlier, duration_sec_scaled]
Index: []
